In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

from sentence_transformers import CrossEncoder
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from openai import OpenAI
import json, time, re
from datetime import datetime
from typing import List, Dict, Any
from qdrant_client import QdrantClient
from langchain_community.embeddings import HuggingFaceEmbeddings
try:
    from ragas import evaluate, RunConfig
    from ragas.metrics import (
        Faithfulness,
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
        AnswerCorrectness,
        AnswerSimilarity
    )
    from ragas.llms       import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from datasets import Dataset
    print("✅ RAGAS berhasil diimpor")
except ImportError as e:
    raise ImportError(f"Import gagal: {e}\n Jalankan: !pip install -q ragas datasets langchain-groq langchain-community")

✅ RAGAS berhasil diimpor


/tmp/ipykernel_14478/2651762360.py:17: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_14478/2651762360.py:17: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/tmp/ipykernel_14478/2651762360.py:17: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
/tmp/ipykernel_14478/2651762360.py:17: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be remo

In [2]:
COLLECTION_NAME    = "Thesis Guide for USU Computer Science Students"              # Nama collection Qdrant
EMBEDDING_MODEL    = "paraphrase-multilingual-MiniLM-L12-v2"

# Retrieval
TOP_K              = 25                      # Jumlah vektor yang di-retrieve
TOP_N_RERANK       = 10                       # Jumlah hasil setelah reranking

# NVIDIA
NVIDIA_MODEL = "meta/llama-3.1-8b-instruct"
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NVIDIA_BASE_URL = os.getenv("NVIDIA_BASE_URL")                                    # (~375 token; 5 chunk × 1500 ≈ 1875 token context)
MAX_TOKENS_OUTPUT   = 2048                     # Maks token jawaban LLM
MAX_CHARS_PER_CHUNK = 1500   

HF_TOKEN = os.getenv("HF_TOKEN")

# Credentials dari .env
QDRANT_URL         = os.getenv("QDRANT_URL")
QDRANT_API_KEY     = os.getenv("QDRANT_API_KEY")

# Validasi
assert QDRANT_URL,     "❌ QDRANT_URL tidak ditemukan di .env!"
assert QDRANT_API_KEY, "❌ QDRANT_API_KEY tidak ditemukan di .env!"
assert NVIDIA_API_KEY,   "❌ NVIDIA_API_KEY tidak ditemukan di .env!"
assert NVIDIA_MODEL,   "❌ NVIDIA_MODEL tidak ditemukan di .env!"
assert NVIDIA_BASE_URL,   "❌ NVIDIA_BASE_URL tidak ditemukan di .env!"


print("✅ Konfigurasi berhasil dimuat!")
print(f"   🤖 NVIDIA_MODEL Model         : {NVIDIA_MODEL}")
print(f"   ✂️  Max chars/chunk    : {MAX_CHARS_PER_CHUNK}")
print(f"   📤 Max tokens output  : {MAX_TOKENS_OUTPUT}")

✅ Konfigurasi berhasil dimuat!
   🤖 NVIDIA_MODEL Model         : meta/llama-3.1-8b-instruct
   ✂️  Max chars/chunk    : 1500
   📤 Max tokens output  : 2048


In [3]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"

# ─── Inisialisasi CrossEncoder Reranker ──────────────────────────────────────
print(f"⏳ Memuat CrossEncoder reranker: {RERANKER_MODEL} ...")
cross_encoder = CrossEncoder(
    model_name=RERANKER_MODEL,
    max_length=512
)
print("✅ CrossEncoder reranker siap!")

The CrossEncoder `model_name` argument was renamed and is now deprecated. Please use `model_name_or_path` instead.


⏳ Memuat CrossEncoder reranker: cross-encoder/ms-marco-MiniLM-L-12-v2 ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ CrossEncoder reranker siap!


In [5]:
nim_client = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url=NVIDIA_BASE_URL  # contoh: "https://integrate.api.nvidia.com/v1"
)

In [6]:
PROMPT_TEMPLATE = """Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Query: {query}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.
Responses should be properly formatted to be easily read."""

In [7]:
EVAL_QUESTIONS: List[str] = [
    # === Bagian 1: Kegiatan Seminar dan Sidang ===
    "Berapa batas minimal dan maksimal jumlah peserta dalam satu kali kegiatan seminar dan sidang?",
    "Jika mahasiswa mengikuti Seminar Proposal pada tanggal 14 April 2026, kapan batas akhir pengumpulan berkasnya?",
    "Sebutkan format penamaan file untuk Proposal dan Form Persetujuan Seminar Proposal yang harus diunggah pada tahap Persetujuan Seminar Proposal!",
    "Apa format nama file submission dan apa saja yang harus disertakan setelah mahasiswa selesai melakukan uji program secara online?",


    # === Bagian 2: Format Penulisan Skripsi ===
    "Berapa ukuran, jenis, dan warna kertas yang diwajibkan untuk salinan naskah asli skripsi?",
    "Apa perbedaan warna kulit depan skripsi antara Program Studi S1 Ilmu Komputer dan Program Studi S1 Teknologi Informasi?",
    "Berapa spesifikasi margin (atas, kanan, kiri, bawah) yang harus digunakan pada halaman awal setiap bab?",
    "Bagaimana aturan penomoran halaman untuk bagian muka skripsi dibandingkan dengan bagian tubuh skripsi?",


    # === Bagian 3: Lampiran dan Detail Teknis ===
    "Berapa ukuran margin tepi kertas yang ditunjukkan pada Lampiran 3 untuk penulisan awal bab?",
    "Apa saja informasi teks yang terdapat pada tulang belakang skripsi S1 Ilmu Komputer berdasarkan Lampiran 2A?",
    "Berdasarkan Lampiran 4A, kalimat apa yang tertulis tepat di bawah tulisan 'SKRIPSI' pada halaman judul skripsi S1 Ilmu Komputer?",
    "Berapa spasi yang digunakan untuk penulisan sub-bab '1.1. Latar Belakang' menurut Lampiran 3?",

    # === Bagian 4: Executive Summary dan Persyaratan ===
    "Apa yang dimaksud dengan Executive Summary (Exum) dan komponen apa saja yang harus ada di dalamnya menurut pedoman?",
    "Bagaimana kriteria penilaian kelayakan Executive Summary (Exum) oleh Kepala Laboratorium dan apa konsekuensi dari masing-masing nilai tersebut?",
    "Apa saja persyaratan akademik dan administratif yang harus dipenuhi mahasiswa untuk dapat mengajukan Seminar Proposal Tugas Akhir?",
    "Bagaimana ketentuan pakaian (dress code) yang wajib dikenakan oleh mahasiswa saat pelaksanaan Seminar Proposal?",

]

GROUND_TRUTHS: List[str] = [
    "Minimal 1 orang dan maksimal 7 orang dalam satu kali kegiatan seminar dan sidang.",
    "Jika seminar proposal dilaksanakan 14 April 2026, batas akhir pengumpulan berkas adalah 08 April 2026, pukul 11.00 WIB.",
    "Format penamaan file: PROPOSAL_NIM.pdf untuk Proposal, dan DOPING_NIM.pdf untuk Form Persetujuan Seminar Proposal.",
    "Format submission: NIM_Judul Tugas Akhir, disertai Link Video Demo Program.",
    
    "Ukuran kertas A4 (210 mm x 297 mm), jenis HVS 70 gram, dan warna kertas putih.",
    "S1 Ilmu Komputer menggunakan warna kulit Hitam, sedangkan S1 Teknologi Informasi menggunakan warna kulit Abu-abu.",
    "Margin awal bab: atas 50 mm, kanan 25 mm, kiri 38 mm, bawah 25 mm dari tepi kertas.",
    "Bagian muka menggunakan angka Romawi kecil (i, ii, iii...), sedangkan bagian tubuh menggunakan angka Arab (1, 2, 3...).",
    
    "Margin pada Lampiran 3: atas 50 mm, kanan 25 mm, kiri 38 mm, bawah 25 mm.",
    "Tulang belakang skripsi S1 Ilmu Komputer memuat: nama mahasiswa, tulisan S1 ILMU KOMPUTER, dan Fasilkom-TI 2012.",
    "Kalimat di bawah 'SKRIPSI': Diajukan untuk melengkapi tugas dan memenuhi syarat memperoleh ijazah Sarjana Ilmu Komputer.",
    "Sub-bab '1.1. Latar Belakang' ditulis dengan jarak 4 spasi.",
    
    "Exum adalah pengajuan ide tugas akhir sebelum proposal, terdiri dari Judul, Latar Belakang, Rumusan Masalah, Metodologi, dan Referensi.",
    "Penilaian Exum: 'layak' (nilai 100) lanjut ke proposal; 'layak dengan perbaikan' (70–99) wajib revisi; 'tidak layak' (<70) wajib ganti judul.",
    "Syarat Seminar Proposal: minimal 100 SKS dengan IPK ≥2.00, telah menyelesaikan PKL/MBKM dan mata kuliah wajib, lulus Metodologi Penelitian, serta proposal telah di-acc kedua pembimbing.",
    "Dress code: atasan putih, blazer hitam, bawahan hitam; mahasiswa laki-laki wajib memakai dasi."
]

assert len(EVAL_QUESTIONS) == len(GROUND_TRUTHS), \
    f"❌ Jumlah pertanyaan ({len(EVAL_QUESTIONS)}) ≠ ground truth ({len(GROUND_TRUTHS)})"

In [8]:
def setup_ragas_judge():
    # ── Judge LLM: NVIDIA NIM ──
    judge_llm = LangchainLLMWrapper(
        ChatOpenAI(
            model=NVIDIA_MODEL,
            openai_api_key=NVIDIA_API_KEY,
            openai_api_base=NVIDIA_BASE_URL,
            temperature=0.01,
            max_tokens=2048,
            n=1,
        )
    )
    # ── Judge Embedding: HuggingFace lokal ──
    judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(
        model_name="paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    ))
    return judge_llm, judge_emb

In [9]:
print(f"⏳ Menghubungkan ke Qdrant di {QDRANT_URL} ...")
qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60
)
print("✅ Koneksi berhasil!")

⏳ Menghubungkan ke Qdrant di https://ea295c4d-3efb-4676-9964-a20d09b4e162.us-east4-0.gcp.cloud.qdrant.io ...
✅ Koneksi berhasil!


In [10]:
def ask_with_context(query: str):
    """
    Sama persis dengan ask() di notebook, tapi juga mengembalikan list contexts
    (dibutuhkan RAGAS untuk menghitung context_precision & context_recall).

    Returns:
        answer   (str)        : jawaban LLM
        contexts (list[str])  : list teks chunk yang masuk ke prompt
    """
    # Embed query
    q_embedding = embed_model.encode(query, normalize_embeddings=True).tolist()

    # Retrieve top-K dari Qdrant
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding,
        limit=TOP_K,
        with_payload=True
    ).points

    candidates = [
        {
            "id"     : h.id,
            "text"   : h.payload["text"],
            "source" : h.payload["source"],
            "score"  : h.score
        }
        for h in hits
    ]

    # CrossEncoder reranking
    pairs         = [[query, c["text"]] for c in candidates]
    rerank_scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, rerank_scores):
        c["rerank_score"] = float(s)

    # Ambil TOP_N_RERANK setelah reranking
    top_reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:TOP_N_RERANK]

    # Contexts sebagai list string (format RAGAS)
    contexts = [r["text"] for r in top_reranked]

    # Potong sesuai MAX_CHARS_PER_CHUNK (sesuai notebook)
    contexts_trimmed = [c[:MAX_CHARS_PER_CHUNK] for c in contexts]

    # Build context string untuk prompt
    context_str = "\n\n".join(
        f"[Source {i+1}: {r['source']}]\n{r['text'][:MAX_CHARS_PER_CHUNK]}"
        for i, r in enumerate(top_reranked)
    )

    # Build prompt (sama dengan notebook)
    prompt = PROMPT_TEMPLATE.format(context=context_str, query=query)

    # Panggil Groq
    resp = nim_client.chat.completions.create(
        model=NVIDIA_MODEL,  # contoh: "meta/llama-3.1-8b-instruct"
        messages=[
            {
                "role"   : "system",
                "content": "You are a helpful assistant that answers questions based on provided context."
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=MAX_TOKENS_OUTPUT,
        top_p=0.9
    )
    answer = resp.choices[0].message.content
    return answer, contexts_trimmed

In [11]:
def collect_rag_outputs(questions: List[str], ground_truths: List[str]) -> Dict[str, List]:
    """Jalankan ask_with_context() untuk setiap pertanyaan evaluasi."""
    all_questions, all_answers, all_contexts, all_gt = [], [], [], []

    print(f"📥 Mengumpulkan output RAG untuk {len(questions)} pertanyaan...\n")

    for i, (question, gt) in enumerate(zip(questions, ground_truths), 1):
        print(f"  [{i}/{len(questions)}] {question[:70]}...")
        try:
            answer, contexts = ask_with_context(question)
            all_questions.append(question)
            all_answers.append(answer)
            all_contexts.append(contexts)
            all_gt.append(gt)
            print(f"         ✅ {len(contexts)} konteks | Jawaban: {answer[:80]}...")
        except Exception as e:
            print(f"         ❌ Error: {e}")
            all_questions.append(question)
            all_answers.append("ERROR")
            all_contexts.append([""])
            all_gt.append(gt)

        time.sleep(1.0)   # hindari rate limit 
        print()

    return {
        "question"     : all_questions,
        "answer"       : all_answers,
        "contexts"     : all_contexts,
        "ground_truth" : all_gt,
    }


In [12]:
def run_ragas_evaluation(rag_data: Dict[str, List]) -> Dict[str, Any]:
    """Jalankan evaluasi RAGAS — kembalikan hasil mentah + DataFrame skor."""
    print("\n" + "="*70)
    print("🧪 MEMULAI EVALUASI RAGAS")
    print("="*70)

    # Setup judge
    print("\n⚙️  Menyiapkan judge LLM & embeddings...")
    judge_llm, judge_emb = setup_ragas_judge()

    # Daftar metrik
    answer_similarity = AnswerSimilarity()
    answer_similarity.embeddings = judge_emb
    
    answer_correctness = AnswerCorrectness()
    answer_correctness.llm = judge_llm
    answer_correctness.embeddings = judge_emb
    answer_correctness.answer_similarity = answer_similarity  # ← inject manual
    
    metrics = [
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
        answer_correctness,  # ← pakai instance yang sudah dikonfigurasi
    ]
    
    for m in metrics:
        m.llm = judge_llm
        if hasattr(m, "embeddings"):
            m.embeddings = judge_emb

    # Buat dataset
    dataset = Dataset.from_dict(rag_data)
    print(f"\n📊 Dataset  : {len(dataset)} sampel")
    print(f"📐 Metrik   : {[type(m).__name__ for m in metrics]}\n")

    run_config = RunConfig(
        timeout=120,
        max_retries=3,
        max_wait=60,
        max_workers=1,
    )

    print("⏳ Menjalankan evaluasi...\n")
    t0 = time.time()
    result = evaluate(
        dataset=dataset,
        metrics=metrics,
        run_config=run_config,
    )
    elapsed = time.time() - t0
    print(f"\n✅ Evaluasi selesai dalam {elapsed:.1f} detik")

    scores_df = result.to_pandas()
    metric_names = [type(m).__name__ for m in metrics]
    return {
        "result"      : result,
        "scores_df"   : scores_df,
        "metric_names": metric_names,
        "elapsed_sec" : round(elapsed, 1),
        "n_samples"   : len(dataset),
    }

In [13]:
def compute_and_save_ragas_scores(eval_output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Hitung skor agregat dari hasil evaluasi RAGAS.
    Kompatibel dengan RAGAS v0.1 (kolom 'question') dan v0.2+ (kolom 'user_input').
    """
    scores_df    = eval_output["scores_df"]
    print("faithfulness =",scores_df["faithfulness"].mean())
    print("answer_relevancy =",scores_df["answer_relevancy"].mean())
    print("context_precision =",scores_df["context_precision"].mean())
    print("context_recall =",scores_df["context_recall"].mean())
    print("answer_correctness =",scores_df["answer_correctness"].mean())
    
    csv_path  = f"ragas_hasil.csv"
    
    scores_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\n💾 Hasil disimpan:")
    print(f"   CSV  : {csv_path}")

    

In [14]:
def _label(score: float) -> str:
    if score >= 0.85: return "🟢 Sangat Baik"
    if score >= 0.70: return "🟡 Baik"
    if score >= 0.55: return "🟠 Cukup"
    return  

In [15]:
# Step 1: Kumpulkan output pipeline
rag_data = collect_rag_outputs(EVAL_QUESTIONS, GROUND_TRUTHS)

# Step 2: Evaluasi RAGAS


📥 Mengumpulkan output RAG untuk 16 pertanyaan...

  [1/16] Berapa batas minimal dan maksimal jumlah peserta dalam satu kali kegia...
         ✅ 10 konteks | Jawaban: Berikut adalah jawaban untuk pertanyaan Anda:

Minimal jumlah peserta dalam satu...

  [2/16] Jika mahasiswa mengikuti Seminar Proposal pada tanggal 14 April 2026, ...
         ✅ 10 konteks | Jawaban: Based on the provided information, I found the relevant table for the Seminar Pr...

  [3/16] Sebutkan format penamaan file untuk Proposal dan Form Persetujuan Semi...
         ✅ 10 konteks | Jawaban: Berikut adalah format penamaan file untuk Proposal dan Form Persetujuan Seminar ...

  [4/16] Apa format nama file submission dan apa saja yang harus disertakan set...
         ✅ 10 konteks | Jawaban: Apa format nama file submission dan apa saja yang harus disertakan setelah mahas...

  [5/16] Berapa ukuran, jenis, dan warna kertas yang diwajibkan untuk salinan n...
         ✅ 10 konteks | Jawaban: Berikut adalah jawaban atas pe

In [16]:
eval_output = run_ragas_evaluation(rag_data)


🧪 MEMULAI EVALUASI RAGAS

⚙️  Menyiapkan judge LLM & embeddings...


/tmp/ipykernel_14478/2924979825.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(
/tmp/ipykernel_14478/2924979825.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_14478/2924979825.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(



📊 Dataset  : 16 sampel
📐 Metrik   : ['Faithfulness', 'AnswerRelevancy', 'ContextPrecision', 'ContextRecall', 'AnswerCorrectness']

⏳ Menjalankan evaluasi...



Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[18]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[21]: OutputParserException(Invalid json output: Ukuran kertas yang diwajibkan untuk salinan naskah asli skripsi adalah A4 (210 mm x 297 mm).
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1


✅ Evaluasi selesai dalam 1432.5 detik


In [17]:
compute_and_save_ragas_scores(eval_output)

faithfulness = 0.7581845238095237
answer_relevancy = 0.8145054544006309
context_precision = 0.5684275793409403
context_recall = 0.775
answer_correctness = 0.636615175011509

💾 Hasil disimpan:
   CSV  : ragas_hasil.csv
